### Starting the silos 2018 dataset

In [29]:
import pandas as pd
import os 
import numpy as np
from sklearn.model_selection import train_test_split
import numpy as np 

In [30]:
data = pd.read_parquet('../datasets/cic-ids2018dataset/datasets/Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter.parquet')
data.head(5)

,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,6,141385,9,7,553,3773.0,202,0,61.444443,87.534439,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1,6,281,2,1,38,0.0,38,0,19.000000,26.870058,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
2,6,279824,11,15,1086,10527.0,385,0,98.727272,129.392502,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
3,6,132,2,0,0,0.0,0,0,0.000000,0.000000,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
4,6,274016,9,13,1285,6141.0,517,0,142.777771,183.887726,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign


In [31]:
### Combining all the datasets 
silo_path = '../datasets/cic-ids2018dataset/datasets/'
datasets = []

for filename in os.listdir(silo_path):
    if filename.endswith('.parquet'):
        filepath = os.path.join(silo_path, filename) ### Creating a file path for each directory
        data = pd.read_parquet(filepath)
        datasets.append(data)
    
combined_datasets = pd.concat(datasets, ignore_index=True)
combined_datasets.head(5)
print(combined_datasets.shape[0])

6659532


In [32]:
print(combined_datasets['Label'].value_counts())

Label
Benign                      5329008
DDoS attacks-LOIC-HTTP       575364
DDOS attack-HOIC             198861
DoS attacks-Hulk             145199
Bot                          144535
Infilteration                118483
SSH-Bruteforce                94048
DoS attacks-GoldenEye         41406
DoS attacks-Slowloris          9908
DDOS attack-LOIC-UDP           1730
Brute Force -Web                568
Brute Force -XSS                229
SQL Injection                    85
DoS attacks-SlowHTTPTest         55
FTP-BruteForce                   53
Name: count, dtype: int64


In [33]:
## Only the attack samples 
attack_data = combined_datasets[combined_datasets['Label'] != 'Benign']
benign_data = combined_datasets[combined_datasets['Label'] == 'Benign']

In [34]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
NUM_CLIENTS = 4
ALPHA = 0.5
client_indices = [[] for _ in range(NUM_CLIENTS)]
for label in attack_data['Label'].unique():    
    indices = attack_data[attack_data['Label'] == label].index.to_numpy().copy() ## This returns the row number of each attack type sample
    np.random.shuffle(indices)
    proportions = np.random.dirichlet([ALPHA] * NUM_CLIENTS)
    split_points = (np.cumsum(proportions)[:-1] * len(indices)).astype(int)
    splits = np.split(indices, split_points)
    for client_id in range(NUM_CLIENTS):
        client_indices[client_id].extend(splits[client_id])

### each silo has sttack indices

In [35]:
client_datasets = []
for client_id in range(NUM_CLIENTS):
    attack_count = len(client_indices[client_id])
    benign_sample = benign_data.sample(n = attack_count, random_state=RANDOM_SEED)
    client_dataset = pd.concat(
    [
        attack_data.loc[client_indices[client_id]],
        benign_sample
    ])
    client_datasets.append(client_dataset)
    

In [36]:
for each in client_datasets:
    print(each.shape)

(1336088, 78)
(290252, 78)
(648204, 78)
(386504, 78)


In [39]:
def binary_silos_converter(df):
    df = df.copy()
    df['Label_Binary'] = df['Label'].apply(lambda x: 0 if x == 'Benign' else 1)
    df = df.drop(columns = 'Label')
    return df 

In [ ]:
pd1 = client_datasets[0]
pd1_binary = binary_silos_converter(pd1)
print(pd1_binary['Label_Binary'].value_counts())
pd1_binary.to_csv('../silos_datasets/silos_datasets2018/SiloBinaryOne.csv')

Label_Binary
1    668044
0    668044
Name: count, dtype: int64
